In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

In [ ]:
#Loading all cleaned datasets
brfss_clean=pd.read_csv("/content/brfss_clean.csv")
uganda_clean=pd.read_csv("/content/uganda_clean.csv")
geda_clean=pd.read_csv("/content/geda_clean.csv")

print("BRFSS:", brfss_clean.shape)
print("Uganda:", uganda_clean.shape)
print("GEDA:", geda_clean.shape)


BRFSS: (429360, 7)
Uganda: (3781, 8)
GEDA: (11354, 7)


In [ ]:
features =["age", "sex_male", "bmi", "hypertension", "current_smoker", "physically_active"]
target ="diabetes_binary"

In [ ]:
def run_logistic_model(df, dataset_name):
    data=df.dropna(subset=[target]).copy()
    x= data[features]
    y=data[target]

    x_train, x_test, y_train, y_test =train_test_split(
    x,y, test_size=0.3, stratify=y, random_state=42
    )

    preprocessor =ColumnTransformer(
         transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), features)

         ]

    )

    model = Pipeline([
       ("prep", preprocessor),
       ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))

    ])

    model.fit(x_train, y_train)

    proba =model.predict_proba(x_test)[:,1]
    pred=(proba >=0.5).astype(int)

    results={
        "Dataset": dataset_name,
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    }

    return results


In [ ]:
brfss_results =run_logistic_model(brfss_clean, "BRFSS")
uganda_results =run_logistic_model(uganda_clean, "Uganda")
geda_results =run_logistic_model(geda_clean, "GEDA")

results_df =pd.DataFrame([brfss_results, uganda_results,geda_results])
results_df


,Dataset,AUC,Precision,Recall,F1
0,BRFSS,0.790771,0.268100,0.761658,0.396599
1,Uganda,0.628482,0.021807,0.500000,0.041791
2,GEDA,0.801359,0.211317,0.761146,0.330796


**1. BRFSS**

The BRFSS internal benchmark shows good discrimination performance with an AUC of 0.791. This suggests that the harmonized baseline pipeline is functioning well on a large well documneted benchmark dataset.

**2. GEDA**

GEDA perfoms slightly better than BRFSS on AUC. This suggests that the German variables are usuable, the GEDA cleaned dataset is good quality.

The internal German model produced the strongest discrimination performance among the 3 datasets with an AUC of 0.801. This shows that the selected harmonized predictors retain useful information for diabetes prediction in the GEDA dataset.

**3. Uganda**

Uganda performs much worse than BRFSS and GEDA because of the much smaller sample size and very small number of diabetes positive cases (only 46) making the classification problem more difficult.




So since GEDA age is grouped, GEDA BMI is grouped, Uganda BMI is continuous, BRFSS BMI is continuous, i should make the variables more comparable before external validation.

So, I willcreate a fully harmonized modelling version where age and BMI are grouped in all datasets, the other variables remain binary.